# AI应用基础 · 工单M2.1《工业焊接缺陷检测系统》
**第6周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 加数据增强（随机翻转/平移）对比训练曲线，解释增强为何有效；示范讲解 |
| **B 组长** | 标准达成 | ≥0.88；数据增强对比+loss曲线 |
| **C 操作员** | 基础完成 | 准确率≥0.80；形状追踪表基本正确 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

## 一、课前资讯（Predict·预测）
> **预习快问**：卷积核3×3、输入64×64、不加padding，输出尺寸是多少？

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m2img"], check=True)
print("数据已生成: data/")

工作目录: D:\Python编程实战\教学实施资料\01-AI应用基础
数据已生成: data/


## 三、任务要求（Make）
**实施步骤**
1. 加载npy数据并归一化(/255)+划分
2. 补全forward与训练循环(3 epochs)
3. 记录每轮loss与测试准确率
4. 输出2张测试图预测+置信度

**验收标准**：及格(D)=`准确率≥0.80；形状追踪表基本正确` ；优秀(B+)=`≥0.88；数据增强对比+loss曲线`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [1]:
# -*- coding: utf-8 -*-
"""M2.1 工业焊接缺陷检测(tiny-CNN, CPU约2分钟) —— 学生代码框架"""
import json
import os

import numpy as np
import torch
import torch.nn as nn

ROOT = "data/weld_images"
CLASSES = ["normal", "porosity", "slag", "crack"]

### Starter 2：`class TinyCNN(nn.Module):`
（Investigate: 读代码→Modify: 补全TODO）

In [1]:
class TinyCNN(nn.Module):
    """2卷积+2全连接（完成态）"""

    def __init__(self, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3)    # 64→62
        self.pool = nn.MaxPool2d(2, 2)     # 62→31
        self.conv2 = nn.Conv2d(8, 16, 3)   # 31→29 →pool→14
        self.fc1 = nn.Linear(16 * 14 * 14, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.flatten(1)
        return self.fc2(torch.relu(self.fc1(x)))

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - （在此处编写你的代码——依据上方注释/文档字符串）
> - TODO-2: 小批量训练(batch=32):
>   前向→loss→反向→step, 累计打印epoch loss
>   （在此处编写你的代码——依据上方注释/文档字符串）
> - TODO-3: 任取2张测试图, 输出预测类别与softmax置信度
>   TODO-4: (优秀)数据增强: 随机水平翻转后再训练, 对比准确率
>   TODO-5: (优秀)绘制loss曲线与混淆矩阵(matplotlib)

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`准确率≥0.80；形状追踪表基本正确`，优秀(B+)=`≥0.88；数据增强对比+loss曲线`

### Starter 3：`def load_dataset():`
（Investigate: 读代码→Modify: 补全TODO）

In [1]:
def load_dataset():
    with open(os.path.join(ROOT, "index.json"), encoding="utf-8") as f:
        meta = json.load(f)
    X = np.stack([np.load(os.path.join(ROOT, m["file"])) for m in meta])
    y = np.array([CLASSES.index(m["label"]) for m in meta])
    X = X.astype(np.float32) / 255.0          # 归一化(勿忘!)
    X = X[:, None, :, :]                      # (N,1,64,64) 通道维
    idx = np.random.RandomState(42).permutation(len(X))
    X, y = X[idx], y[idx]
    n_train = int(len(X) * 0.8)
    return (X[:n_train], y[:n_train]), (X[n_train:], y[n_train:])

### Starter 4：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [1]:
def main():
    (Xtr, ytr), (Xte, yte) = load_dataset()
    print("形状追踪: 输入", Xtr.shape, "(N,C,H,W)")
    device = "cpu"
    model = TinyCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1.5e-3)
    lossf = nn.CrossEntropyLoss()

    xt, yt = torch.tensor(Xte), torch.tensor(yte)
    xtr, ytr_t = torch.tensor(Xtr), torch.tensor(ytr)
    for epoch in range(6):
        model.train()
        perm = torch.randperm(len(xtr))
        tot = 0.0
        for i in range(0, len(xtr), 32):
            b = perm[i:i + 32]
            opt.zero_grad()
            loss = lossf(model(xtr[b]), ytr_t[b])
            loss.backward()
            opt.step()
            tot += loss.item() * len(b)
        model.eval()
        with torch.no_grad():
            pred = model(xt).argmax(1)
            acc = (pred == yt).float().mean().item()
        print(f"epoch {epoch + 1} loss={tot / len(xtr):.4f} 测试准确率={acc:.3f}")

    model.eval()
    with torch.no_grad():
        pr = model(xt).argmax(1)
        for k in range(2):
            conf = torch.softmax(model(xt[k:k + 1]), 1).max().item()
            print(f"样本{k}: 预测={CLASSES[pr[k]]} 置信度={conf:.3f}")
    print("形状追踪: (N,1,64,64)→conv1(N,8,62,62)→pool(31)→conv2(N,16,29,29)"
          "→pool(14)→flatten(N,3136)→fc→(N,4)")

### Starter 5：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [1]:
if __name__ == "__main__":
    main()

形状追踪: 输入 (192, 1, 64, 64) (N,C,H,W)
epoch 1 loss=1.3948 测试准确率=0.208
epoch 2 loss=1.3824 测试准确率=0.292
epoch 3 loss=1.3683 测试准确率=0.354
epoch 4 loss=1.3373 测试准确率=0.521
epoch 5 loss=1.2803 测试准确率=0.479
epoch 6 loss=1.1739 测试准确率=0.500
样本0: 预测=porosity 置信度=0.316
样本1: 预测=slag 置信度=0.355
形状追踪: (N,1,64,64)→conv1(N,8,62,62)→pool(31)→conv2(N,16,29,29)→pool(14)→flatten(N,3136)→fc→(N,4)


## 四、提交要求与避坑
- 提交: `学号_姓名_M2.1.py` 或本notebook(含输出)
- ⚠ 输入必须/255归一化，否则loss居高不下
- ⚠ PyTorch通道序是N,C,H,W——numpy转tensor后要补通道维
- ⚠ 测试集指标为准，禁止在测试集上挑图刷准确率

---

## 教师参考答案（来源：`教师参考/solutions.py` · `m21()`）

> **发放纪律：本附录勿随学生版下发。**

上方学生版 TODO 与下方参考实现的对应关系：

| 学生版函数 | 说明 |
|---|---|
| — | 本课无 TODO |

In [1]:
def m21():
    import torch
    import torch.nn as nn

    with open("data/weld_images/index.json", encoding="utf-8") as f:
        meta = json.load(f)
    X = np.stack([np.load("data/weld_images/" + m["file"]) for m in meta])
    y = np.array([["normal", "porosity", "slag", "crack"].index(m["label"])
                  for m in meta])
    X = (X.astype(np.float32) / 255.0)[:, None]
    idx = np.random.RandomState(42).permutation(len(X))
    X, y = X[idx], y[idx]
    n = int(len(X) * 0.8)
    Xtr, ytr = torch.tensor(X[:n]), torch.tensor(y[:n])
    Xte, yte = torch.tensor(X[n:]), torch.tensor(y[n:])

    class TinyCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 8, 3)
            self.pool = nn.MaxPool2d(2, 2)
            self.conv2 = nn.Conv2d(8, 16, 3)
            self.fc1 = nn.Linear(16 * 14 * 14, 64)
            self.fc2 = nn.Linear(64, 4)

        def forward(self, x):
            x = self.pool(torch.relu(self.conv1(x)))
            x = self.pool(torch.relu(self.conv2(x)))
            x = x.flatten(1)
            return self.fc2(torch.relu(self.fc1(x)))

    torch.manual_seed(42)
    model = TinyCNN()
    opt = torch.optim.Adam(model.parameters(), lr=1.5e-3)
    lossf = nn.CrossEntropyLoss()
    for ep in range(6):
        model.train()
        perm = torch.randperm(len(Xtr))
        tot = 0.0
        for i in range(0, len(Xtr), 32):
            b = perm[i:i + 32]
            opt.zero_grad()
            loss = lossf(model(Xtr[b]), ytr[b])
            loss.backward()
            opt.step()
            tot += loss.item() * len(b)
        model.eval()
        with torch.no_grad():
            acc = (model(Xte).argmax(1) == yte).float().mean().item()
        print(f"epoch{ep + 1} loss={tot / len(Xtr):.4f} 测试acc={acc:.3f}")
    print("形状追踪: (N,1,64,64)→conv1(N,8,62,62)→pool(31)→conv2(N,16,29,29)"
          "→pool(14)→flatten(N,3136)→fc→(N,4)")

## 运行与验证

上方参考实现可直接运行（合成数据、CPU 离线、结果可复现）。
多解法对比见《多解法手册.md》。